# 03 - Inference with the Packaged Model
Three ways to use the champion: the joblib pipeline (used by the API), the MLflow-format model, and the REST API.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)

Project root: D:\BITS\Semester-3\MLOps\MLOPS_Webinar1\heart-disease-mlops


In [2]:
import json
from src.inference.predict import load_model_bundle, predict_records
model, metadata = load_model_bundle()
{k: metadata[k] for k in ['model_name','model_family','model_version','trained_at','test_metrics']}

{'model_name': 'HeartDiseaseRiskChampion',
 'model_family': 'logistic_regression',
 'model_version': '2',
 'trained_at': '2026-10-13T17:03:29.942632+00:00',
 'test_metrics': {'accuracy': 0.8688524590163934,
  'precision': 0.8125,
  'recall': 0.9285714285714286,
  'f1': 0.8666666666666667,
  'roc_auc': 0.9577922077922079}}

In [3]:
patient = json.load(open('scripts/sample_request.json'))
predict_records(model, [patient, {**patient, 'cp': 4, 'exang': 1, 'ca': 2, 'thal': 7, 'thalach': 110}])

[{'prediction': 0,
  'label': 'no_heart_disease',
  'confidence': 0.7385,
  'probability_heart_disease': 0.2615,
  'risk_level': 'low'},
 {'prediction': 1,
  'label': 'heart_disease',
  'confidence': 0.9909,
  'probability_heart_disease': 0.9909,
  'risk_level': 'high'}]

## MLflow pyfunc (portable format)

In [4]:
import mlflow.pyfunc
pyfunc_model = mlflow.pyfunc.load_model('models/mlflow_model')
pyfunc_model.predict(pd.DataFrame([patient]).astype('float64'))

array([0])

## Batch scoring

In [5]:
from src.inference.predict import run_batch_inference
scored = run_batch_inference()
print('agreement with labels:', round((scored.prediction == scored.target).mean(), 3))
scored[['age','cp','thalach','target','prediction','probability_heart_disease','risk_level']].head(10)

agreement with labels: 0.861


,age,cp,thalach,target,prediction,probability_heart_disease,risk_level
0,63.0,1.0,150.0,0,0,0.2615,low
1,67.0,4.0,108.0,1,1,0.9961,high
2,67.0,4.0,129.0,1,1,0.9911,high
3,37.0,3.0,187.0,0,0,0.1158,low
4,41.0,2.0,172.0,0,0,0.0550,low
5,56.0,2.0,178.0,0,0,0.0722,low
6,62.0,4.0,160.0,1,1,0.7919,high
7,57.0,4.0,163.0,0,0,0.1474,low
8,63.0,4.0,147.0,1,1,0.9437,high
9,53.0,4.0,155.0,1,1,0.8424,high


## REST API (start it first: `uvicorn src.api.main:app --port 8000` or `docker run -p 8000:8000 heart-disease-api`)

In [6]:
import requests
try:
    r = requests.post('http://127.0.0.1:8000/predict', json=patient, timeout=3)
    print(r.status_code, r.json())
except requests.exceptions.ConnectionError:
    print('API not running - start it to try this cell.')

API not running - start it to try this cell.
